# Exploração — Camada Raw (dbt)

Queries diretas sobre as 13 views no schema `raw` do DuckDB.  
Arquivo: `dbt/geo_analytics.duckdb` | Schema: `raw`

> As views são réplicas fieis dos Parquets — schema idêntico ao dado bruto da ingestão.

> ⚠️ **Antes de rodar `dbt run`:** execute a célula de cleanup no final do notebook (`conn.close()`) ou reinicie o kernel. O DuckDB não permite conexão write-mode enquanto este notebook estiver conectado.

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)

conn = duckdb.connect('../dbt/geo_analytics.duckdb', read_only=True)

def q(sql):
    return conn.execute(sql).fetchdf()

print('conectado')

conectado


## Catálogo

In [5]:
# Todas as views no schema raw
q("""
    select view_name, sql
    from duckdb_views()
    where schema_name = 'raw'
""")

,view_name,sql
0,bcb_pix,CREATE VIEW raw.bcb_pix AS SELECT * ...
1,ibge_censo_9514,CREATE VIEW raw.ibge_censo_9514 AS S...
2,ibge_censo_9605,CREATE VIEW raw.ibge_censo_9605 AS S...
3,ibge_censo_9606,CREATE VIEW raw.ibge_censo_9606 AS S...
4,ibge_localidades,CREATE VIEW raw.ibge_localidades AS ...
5,olist_customers,CREATE VIEW raw.olist_customers AS S...
6,olist_geolocation,CREATE VIEW raw.olist_geolocation AS...
7,olist_orders,CREATE VIEW raw.olist_orders AS SELE...
8,olist_order_items,CREATE VIEW raw.olist_order_items AS...
9,olist_order_payments,CREATE VIEW raw.olist_order_payments...


In [6]:
# Volume por tabela
tabelas = [
    'olist_customers', 'olist_orders', 'olist_order_items',
    'olist_order_payments', 'olist_order_reviews', 'olist_geolocation',
    'olist_products', 'olist_sellers',
    'ibge_localidades', 'ibge_censo_9606', 'ibge_censo_9605', 'ibge_censo_9514',
    'bcb_pix',
]
rows = [(t, conn.execute(f'select count(*) from raw.{t}').fetchone()[0]) for t in tabelas]
pd.DataFrame(rows, columns=['tabela', 'linhas'])

,tabela,linhas
0,olist_customers,99441
1,olist_orders,99441
2,olist_order_items,112650
3,olist_order_payments,103886
4,olist_order_reviews,99224
5,olist_geolocation,1000163
6,olist_products,32951
7,olist_sellers,3095
8,ibge_localidades,5571
9,ibge_censo_9606,11140


---
## Olist

In [7]:
q('select * from raw.olist_orders limit 5')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,day,month,year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,05,06,2026
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,05,06,2026
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,05,06,2026
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,05,06,2026
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,05,06,2026


In [ ]:
q("""
    select order_status, count(*) as n
    from raw.olist_orders
    group by 1
    order by 2 desc
""")

In [ ]:
q('select * from raw.olist_customers limit 5')

In [ ]:
q('select * from raw.olist_order_items limit 5')

In [ ]:
q('select * from raw.olist_order_payments limit 5')

In [ ]:
q('select * from raw.olist_order_reviews limit 5')

In [ ]:
# Geolocation: múltiplas linhas por zip — sem dedup nesta camada
q("""
    select geolocation_zip_code_prefix, count(*) as n
    from raw.olist_geolocation
    group by 1
    order by 2 desc
    limit 10
""")

In [ ]:
q('select * from raw.olist_products limit 5')

In [ ]:
q('select * from raw.olist_sellers limit 5')

---
## IBGE

In [ ]:
q('select * from raw.ibge_localidades limit 5')

In [ ]:
# 9606 — chaves internas SIDRA: D4=Sexo, D5=Cor/Raça, D6=Idade
q('select * from raw.ibge_censo_9606 limit 5')

In [ ]:
# 9605 — D5/D6 são NULL (tabela só tem 4 dimensões)
q('select * from raw.ibge_censo_9605 limit 5')

In [ ]:
# 9514 — D4=Sexo, D5=Forma declaração idade, D6=Idade
q('select * from raw.ibge_censo_9514 limit 5')

---
## BCB PIX

In [ ]:
q('select * from raw.bcb_pix limit 5')

In [ ]:
# Cobertura temporal
q("""
    select min(AnoMes) as primeiro_mes, max(AnoMes) as ultimo_mes,
           count(distinct AnoMes) as meses,
           count(distinct Municipio_Ibge) as municipios
    from raw.bcb_pix
""")

---
## Queries livres

In [ ]:
q("""
    -- escreva aqui
""")

---
## Cleanup

Execute antes de rodar `dbt run` — libera o lock do DuckDB.

In [ ]:
conn.close()
print('conexão encerrada')